# Datathon 2026 — Gemini Metin Anotasyonu (v2, düzeltilmiş)

20 bin mentor yorumunu Gemini'ye okutup yapılandırılmış özellik çıkarır
(ima edilen puan, ton, güçlü/zayıf yönler, kesinlik).

**Kurulum:**
1. https://aistudio.google.com/apikey → ücretsiz API anahtarı → aşağıda `API_KEY`e yapıştırın
2. Drive `MyDrive/datathon/` içinde `train.csv` + `test.csv`
3. Tümünü çalıştır — **GPU gerekmez**, ~1.5-2 saat

**Düzeltmeler (v2):** başarısız parti checkpoint'lenmez (eksik veri kalıcılaşmaz),
eksik/bozuk checkpoint yeniden işlenir, JSON modu açık, alan doğrulama/kırpma var,
türetilmiş etkileşim özellikleri eklenir.

**Takip:** ~5. dakikada `>>> ERKEN SINYAL corr=...` satırı düşer — bu değeri Claude'a yazın
(0.70+ = çok güçlü; <0.50 = katkı sınırlı). Kopma olursa Run All ile devam eder.

In [ ]:

API_KEY = "BURAYA_API_ANAHTARI"   # https://aistudio.google.com/apikey — ciktilarda paylasmayin

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','google-genai'])
from google import genai
import glob, os, json, time
import numpy as np, pandas as pd

from google.colab import drive
drive.mount('/content/drive')
CKPT='/content/drive/MyDrive/datathon_gemini'; os.makedirs(CKPT, exist_ok=True)

paths = glob.glob('/content/train.csv') + glob.glob('/content/drive/MyDrive/**/train.csv', recursive=True)
assert paths, 'train.csv bulunamadi: Drive/datathon klasorune koyun'
DATA = os.path.dirname(paths[0])
tr = pd.read_csv(f'{DATA}/train.csv', encoding='utf-8-sig')
tep = f'{DATA}/test.csv' if os.path.exists(f'{DATA}/test.csv') else glob.glob(f'{DATA}/test*.csv')[0]
te = pd.read_csv(tep, encoding='utf-8-sig')
y = tr['career_success_score'].values
print('VERI OK:', tr.shape, te.shape)

client = genai.Client(api_key=API_KEY)
ASPECTS = ['kodlama','problem_cozme','veri_yapilari','sql','makine_ogrenmesi','backend',
           'frontend','bulut','devops','proje_kalitesi','staj','github_acik_kaynak',
           'portfolyo','iletisim','takim_calismasi','liderlik','sunum','mulakat']
PROMPT = ("Asagida mentor degerlendirme yorumlari var. Her yorum icin yorumun tonuna/icerigine gore: "
 '"puan" (ima edilen kariyer skoru 0-100; cok ovgu=85-100, dengeli=55-80, elestirel=20-55), '
 '"ton" (-2..+2), "guclu" ve "zayif" (su listeden: ' + ', '.join(ASPECTS) + '), '
 '"kesinlik" (1-5). SADECE JSON dizisi ver: '
 '[{"id":int,"puan":int,"ton":int,"guclu":[],"zayif":[],"kesinlik":int},...]\n\nYorumlar:\n')

def batch(ids, texts):
    items = '\n'.join(f'[id={i}] {t}' for i, t in zip(ids, texts))
    valid_aspects = set(ASPECTS)
    for attempt in range(6):
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash', contents=PROMPT + items,
                config={'temperature': 0, 'max_output_tokens': 8000,
                        'response_mime_type': 'application/json'})
            arr = json.loads(response.text)
            out = {}
            for obj in arr:
                if 'id' not in obj: continue
                idx = int(obj['id'])
                if idx not in ids: continue
                obj['puan'] = int(np.clip(obj.get('puan', 50), 0, 100))
                obj['ton'] = int(np.clip(obj.get('ton', 0), -2, 2))
                obj['kesinlik'] = int(np.clip(obj.get('kesinlik', 3), 1, 5))
                obj['guclu'] = [x for x in obj.get('guclu', []) if x in valid_aspects]
                obj['zayif'] = [x for x in obj.get('zayif', []) if x in valid_aspects]
                out[idx] = obj
            if len(out) == len(ids): return out
            print(f'Eksik yanit: {len(set(ids)-set(out))}/{len(ids)} | deneme {attempt+1}/6')
        except Exception as e:
            print(f'Gemini hatasi, deneme {attempt+1}/6: {str(e)[:120]}')
        time.sleep(min(60, 5 * (2 ** attempt)))
    return {}

def run(name, texts):
    B = 20; n = len(texts); res = {}
    total = (n + B - 1) // B
    for b in range(total):
        start, end = b*B, min((b+1)*B, n)
        ids = list(range(start, end))
        ck = f'{CKPT}/{name}_b{b}.json'
        if os.path.exists(ck):
            try:
                saved = {int(k): v for k, v in json.load(open(ck)).items()}
                if all(i in saved for i in ids):
                    res.update(saved); continue
                print(f'{name} batch {b}: eksik checkpoint, tekrar islenecek')
            except Exception:
                print(f'{name} batch {b}: bozuk checkpoint, tekrar islenecek')
        out = batch(ids, [str(t)[:600] for t in texts[start:end]])
        if len(out) == len(ids):
            with open(ck, 'w', encoding='utf-8') as f:
                json.dump(out, f, ensure_ascii=False)
            res.update(out)
        else:
            print(f'UYARI: {name} batch {b} tamamlanamadi, checkpoint kaydedilmedi.')
        if (b+1) % 10 == 0 or b == total-1:
            print(f'{name}: parti {b+1}/{total} | anotasyon: {len(res)}/{n}', flush=True)
            if name == 'train' and len(res) > 150:
                ks = sorted(k for k, v in res.items() if v.get('puan') is not None)
                p = np.array([res[k]['puan'] for k in ks], dtype=float)
                if len(p) > 1 and np.std(p) > 0:
                    print(f'   >>> ERKEN SINYAL corr={np.corrcoef(p, y[ks])[0,1]:.4f} '
                          f'| ham MSE={np.mean((p-y[ks])**2):.2f}', flush=True)
        time.sleep(1)
    return res

def feats(res, n):
    rows = []
    for i in range(n):
        o = res.get(i, {})
        r = {'llm_puan': o.get('puan', np.nan), 'llm_ton': o.get('ton', np.nan),
             'llm_kesinlik': o.get('kesinlik', np.nan),
             'llm_n_guclu': len(o.get('guclu', [])), 'llm_n_zayif': len(o.get('zayif', []))}
        for a in ASPECTS:
            r[f'llm_g_{a}'] = int(a in o.get('guclu', []))
            r[f'llm_z_{a}'] = int(a in o.get('zayif', []))
        rows.append(r)
    F = pd.DataFrame(rows)
    F['llm_net_aspect'] = F['llm_n_guclu'] - F['llm_n_zayif']
    F['llm_puan_x_kesinlik'] = F['llm_puan'] * F['llm_kesinlik']
    F['llm_ton_x_kesinlik'] = F['llm_ton'] * F['llm_kesinlik']
    F['llm_puan_centered'] = F['llm_puan'] - 50
    return F

res_tr = run('train', tr['mentor_feedback_text'].fillna('').values)
F = feats(res_tr, len(tr)); F.to_csv(f'{CKPT}/llm_feats_train.csv', index=False)
ok = F['llm_puan'].notna()
print('='*50)
print('TRAIN BITTI | kapsam %.1f%% | corr(llm_puan,y)=%.4f | ham MSE=%.1f' % (
  100*ok.mean(), np.corrcoef(F.loc[ok,'llm_puan'], y[ok.values])[0,1],
  ((F.loc[ok,'llm_puan']-y[ok.values])**2).mean()))
res_te = run('test', te['mentor_feedback_text'].fillna('').values)
feats(res_te, len(te)).to_csv(f'{CKPT}/llm_feats_test.csv', index=False)
print('HEPSI BITTI -> Drive/datathon_gemini/ icindeki llm_feats_train.csv + llm_feats_test.csv dosyalarini Claude\'a yukleyin')
